In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 라이브러리

In [2]:
# 데이터 전처리
import pandas as pd
import numpy as np
import gzip
import json
import copy

# 작업 진행률 시각화
from tqdm import tqdm
tqdm.pandas()

# Torch 모델 구조
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# BERT Model
# Fine Tuning
from transformers import AutoModel, AutoTokenizer
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
from transformers.modeling_outputs import SequenceClassifierOutput


# sklearn 전처리, 데이터 분할, 성능 평가지표
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

# Utils

In [3]:
def extract_cls_embeddings(texts, tokenizer, model, batch_size=16):
    embeddings = []
    model.eval()
    device = model.device  # 모델이 할당된 디바이스 가져오기
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc="Extracting [CLS] embeddings"):
            batch_texts = texts[i:i + batch_size]
            inputs = tokenizer(batch_texts, padding=True, truncation=True, return_tensors="pt")
            inputs = {key: val.to(device) for key, val in inputs.items()}  # 모든 입력을 모델의 디바이스로 이동
            outputs = model(**inputs)
            cls_batch = outputs.last_hidden_state[:, 0, :]
            embeddings.append(cls_batch.cpu().numpy())  # GPU 메모리를 절약하기 위해 CPU로 이동 후 numpy 변환
    return np.vstack(embeddings)

In [4]:
def extract_first_image_urls(images_list):
    # 비어있거나 잘못된 값 방지
    if not images_list or not isinstance(images_list, list) or len(images_list) == 0:
        return pd.Series([None, None, None])
    img = images_list[0]
    return pd.Series([
        img.get('small_image_url'),
        img.get('medium_image_url'),
        img.get('large_image_url')
    ])

# 데이터셋 로딩

In [5]:
# 데이터 로딩
df = pd.read_pickle('/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Cell_Phones_RQ_v1.pkl')
print(df.shape)
df.head()

(23491, 6)


,user_id,asin,text,rating,title_y,medium_image_url
0,AHUPTBY3F3UN2S5H7K5JLP6MAV5Q,B0BJKYJR3L,"Nice privacy shield. Good color, fits well, et...",3,Jephuwneh 2 Pack Gradient Tempered Glass for i...,https://m.media-amazon.com/images/I/515oPLx13k...
1,AHUPTBY3F3UN2S5H7K5JLP6MAV5Q,B0BHTZQHQW,"This thing is great. It's not too big, has a d...",5,"NEOCOR Magnetic Wireless Power Bank, 10,000mAh...",https://m.media-amazon.com/images/I/71K8BDxzsA...
2,AHUPTBY3F3UN2S5H7K5JLP6MAV5Q,B0B9N4VRB9,This is a much better way to mount my iPhone (...,5,Lamicall Gooseneck Phone Holder Bed - [Newest ...,https://m.media-amazon.com/images/I/71b+Sx-Jq7...
3,AHUPTBY3F3UN2S5H7K5JLP6MAV5Q,B0B93H178F,"This is a fantastic charger, especially if you...",5,"Shargeek USB C Charger 140W Wall Charger, Mult...",https://m.media-amazon.com/images/I/61R6UXrAKT...
4,AFERCDY2EFJKT7QUQ75GISNHTFOQ,B0BPXZWCBG,There never seems to be enough charging cords ...,4,"Charging Station for Multiple Devices,60W 12A ...",https://m.media-amazon.com/images/I/51fqzioGKe...


# Load Model

## Sentence-Transformer

In [6]:
# load roberta
tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
model = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

In [7]:
# roberta embedding 추출 후 concat
text_embeddings = extract_cls_embeddings(df['text'].tolist(), tokenizer, model, batch_size=16)
text_embeddings = pd.DataFrame({'sentence_transformer_emb': [e for e in text_embeddings]})
df = pd.concat([df, text_embeddings], axis=1)

Extracting [CLS] embeddings: 100%|██████████| 1469/1469 [09:44<00:00,  2.51it/s]


In [8]:
# 최종 Text Emb Dataset 저장
df.to_pickle('/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Cell_Phones_lgm_rec_v1.pkl')